## 21. Gaussian Copula — Calibrating the Correlation Parameter (ρ)

In [ ]:
# ============================================================
# CELL 1 — Build the monthly default rate time series
# ============================================================
# Reuse Week 1's survival_df and issue dates to build a monthly panel of
# "how many loans were alive this month, and how many of them defaulted
# this specific month" -- the same kind of monthly reconstruction used in
# the Section 13 transition matrix work.

issue_dates_full = pd.to_datetime(df_cleaned['issue_d'], format='mixed')

monthly_records = []
sample_n = min(100000, len(survival_df))  # cap for memory safety
survival_sample = survival_df.sample(n=sample_n, random_state=42)

for idx, row in survival_sample.iterrows():
    issue = issue_dates_full.reindex(survival_sample.index).loc[idx]
    if pd.isna(issue):
        continue
    duration = int(row['duration_months'])
    issue_month = issue.to_period('M').to_timestamp()
    for m in range(duration):
        current_month = (issue_month + pd.DateOffset(months=m)).to_period('M').to_timestamp()
        is_default_month = (m == duration - 1) and (row['default'] == 1)
        monthly_records.append({'month': current_month, 'defaulted': int(is_default_month)})

monthly_panel = pd.DataFrame(monthly_records)
print(f"Monthly panel rows (loan-months): {len(monthly_panel):,}")

In [ ]:
# ============================================================
# CELL 2 — Aggregate into a monthly default rate series
# ============================================================
monthly_default_rate = monthly_panel.groupby('month').agg(
    n_loans=('defaulted', 'count'),
    n_defaults=('defaulted', 'sum')
)
monthly_default_rate['default_rate'] = monthly_default_rate['n_defaults'] / monthly_default_rate['n_loans']

# Drop months with very few loans alive -- too noisy to be meaningful
monthly_default_rate = monthly_default_rate[monthly_default_rate['n_loans'] >= 100]

print(f"Months retained (n_loans >= 100): {len(monthly_default_rate)}")
print(monthly_default_rate['default_rate'].describe())

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_default_rate.index, monthly_default_rate['default_rate'], color='#c0392b')
ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-01'), color='gray', alpha=0.2, label='Recession')
ax.set_title('Observed Monthly Default Rate Over Time')
ax.set_ylabel('Default Rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL — Use Basel-prescribed rho, document the data-calibration
# attempt as deferred (not abandoned)
# ============================================================

# The data-calibrated rho attempt was paused before completion: the observed
# monthly default-rate series shows a strong rising-volatility pattern from
# 2016 onward that visually matches the same seasoning/vintage bias diagnosed
# in Section 17 (recent, unseasoned loan vintages disproportionately show only
# their fast failures in a resolved-loan dataset, inflating apparent
# variance). This was flagged as a likely contaminant before calibrating,
# but the variance-contribution check to confirm/quantify it was deferred
# rather than completed.

# Using the Basel III-prescribed correlation for "other retail" exposures
# as the working value for this simulation, with the data-driven calibration
# left as a documented follow-up rather than abandoned.

rho = 0.12  # Basel-prescribed retail asset correlation, typical anchor in the 0.03-0.16 range

print(f"Using rho = {rho} (Basel III retail exposure anchor)")
print(f"Data-driven calibration attempt: PAUSED, not completed -- pending a")
print(f"check of how much the post-2016 seasoning pattern inflates the")
print(f"observed variance. See note above.")

In [ ]:
# ============================================================
# CELL — TODO marker for the deferred calibration check, to come back to
# ============================================================
"""
TODO (deferred): Confirm whether the post-2016 portion of
monthly_default_rate is inflating variance enough to materially bias a
data-calibrated rho, by comparing var() on the full series vs. a series
restricted to pre-2016 months. If the recent period is found to inflate
variance by roughly 2x or more, recalibrate rho on the restricted series
and compare against the Basel anchor used below before finalizing which
value the Monte Carlo simulation should use.
"""